## Run test: Qwen/Qwen3-VL-8B-Instruct

Aligned with the Hugging Face generated direct-load/Colab-style snippet where possible. Inference Providers snippets are ignored because they do not validate local AMD GPU loading. The direct path uses AutoModelForMultimodalLM; the README also supports the concrete Qwen3VLForConditionalGeneration class.

In [ ]:
%pip install -U transformers accelerate

In [ ]:
# HF Colab aligned image-text-to-text pipeline smoke test on one visible GPU.
import gc
import torch
from PIL import Image
from transformers import AutoProcessor, pipeline

model_path = "Qwen/Qwen3-VL-8B-Instruct"

test_image = "/tmp/qwen_ci_smoke.jpg"
Image.new("RGB", (32, 32), color=(220, 40, 40)).save(test_image)
messages = [{'role': 'user',
  'content': [{'type': 'image', 'url': test_image},
              {'type': 'text', 'text': 'Describe the image briefly.'}]}]
processor = AutoProcessor.from_pretrained(model_path, min_pixels=256, max_pixels=1024)
pipe = pipeline("image-text-to-text", model=model_path, processor=processor, trust_remote_code=True)
pipe_result = pipe(messages, max_new_tokens=8, do_sample=False, use_cache=False, logits_to_keep=1)
print(pipe_result)
pipe_tensor_devices = sorted({str(p.device) for p in pipe.model.parameters()})
print("pipe parameter devices =", pipe_tensor_devices)

del pipe, processor, pipe_result, pipe_tensor_devices
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()


In [ ]:
# HF Colab aligned direct image inference smoke test.
import torch
from transformers import AutoProcessor, AutoModelForMultimodalLM

processor = AutoProcessor.from_pretrained(model_path, min_pixels=256, max_pixels=1024)
model = AutoModelForMultimodalLM.from_pretrained(model_path, device_map="auto", torch_dtype="auto").eval()

messages = [{'role': 'user',
  'content': [{'type': 'image', 'url': test_image},
              {'type': 'text', 'text': 'Describe the image briefly.'}]}]
inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to("cuda")

with torch.inference_mode():
    outputs = model.generate(**inputs, max_new_tokens=8, do_sample=False, use_cache=False, logits_to_keep=1)
print(processor.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True))
